# 04 Evaluation setup and preregistered protocol

This notebook creates one **frozen semantic evaluator** and records the evaluation protocol before inspecting interpolation results. The evaluator is trained and validated only on stratified subsets of the official FashionMNIST training split. The official test split is not used for training or checkpoint selection. Notebook 03 uses it separately for descriptive flow-objective diagnostics; notebooks 05 and 06 use it for the semantic trajectory evaluation.

The flow model is not selected or tuned here. Notebook 05 loads the fixed `medium` Sneaker (7) -> Ankle Boot (9) best-validation checkpoint.

In [ ]:
from pathlib import Path
import json
import os
import sys
import torch

from notebooks.utils.colab_paths import find_repository_root

repo_root = find_repository_root()
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Repository: {repo_root}')
print(f'Device: {device}')

In [ ]:
from api.evaluation_api import load_eval_classifier, train_eval_classifier

EVALUATOR_PATH = repo_root / 'assets' / 'evaluation' / 'fmnist_eval_classifier_v1.pt'
EVALUATOR_CONFIG = {
    'epochs': 10,
    'batch_size': 256,
    'seed': 2026,
    'validation_fraction': 0.1,
}

if EVALUATOR_PATH.is_file():
    evaluator_payload = torch.load(EVALUATOR_PATH, map_location='cpu')
    saved_config = evaluator_payload.get('config', {})
    for key, expected_value in EVALUATOR_CONFIG.items():
        if saved_config.get(key) != expected_value:
            raise ValueError(
                f'Evaluator checkpoint has {key}={saved_config.get(key)!r}; '
                f'expected {expected_value!r}. Use a new versioned checkpoint path '
                'if intentionally changing the evaluator protocol.'
            )
    evaluator = load_eval_classifier(EVALUATOR_PATH, device)
    evaluator_history = evaluator_payload.get('history', [])
    print(f'Loaded frozen evaluator: {EVALUATOR_PATH}')
else:
    evaluator, evaluator_history = train_eval_classifier(
        data_root=repo_root / 'data',
        device=device,
        save_path=EVALUATOR_PATH,
        **EVALUATOR_CONFIG,
    )
    print(f'Trained and saved frozen evaluator: {EVALUATOR_PATH}')

evaluator.eval()
if evaluator_history:
    best_validation_accuracy = max(
        row['validation_accuracy'] for row in evaluator_history
    )
    print(f'Best held-out training-split accuracy: {best_validation_accuracy:.4f}')
print('Evaluator is in eval mode; it will not be fine-tuned on test trajectories.')

## Operationalization and limits

| Research criterion | Evidence used in the evaluation | Interpretation limit |
|---|---|---|
| Plausibility | Side-by-side visual inspection; maximum classifier confidence as a weak proxy | A classifier confidence score is **not** an in-distribution or realism score |
| Semantic progression | Target-class confidence curve and proportion of sample paths that are monotonic (allowing 0.01 probability jitter) | Confidence is a class-level proxy, not an explicit shoe-height or silhouette attribute |
| Target alignment / consistency | Target confidence change from start to end and its path-wise monotonicity | The endpoint is supplementary; target-informed baselines have an oracle advantage |
| Non-target preservation | Source-class confidence curve and non-negative endpoint drop | Class confidence is not a measure of preserving individual, non-target attributes |
| Smoothness | Consecutive probability-vector changes and pixel-space step norms; visual path inspection | Smoothness supports, but does not establish, semantic meaningfulness |

- All methods use the same fixed set of **64 paired official-test examples** and 50 Euler steps
- Linear interpolation uses the paired test target and is labelled oracle-style
- NN-matched targets and class means are drawn only from the official training split
- Bootstrap intervals describe uncertainty over sampled test pairs; they do not resolve limitations of the proxies!!

In [ ]:
from datetime import datetime, timezone

PROTOCOL = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'flow_checkpoint': 'assets/fmnist_d2d_runs/fmnist_d2d_training_checkpoint_best.pt',
    'flow_model_profile': 'medium',
    'flow_pair': {'source': 7, 'target': 9},
    'evaluator_checkpoint': str(EVALUATOR_PATH.relative_to(repo_root)),
    'evaluator_config': EVALUATOR_CONFIG,
    'evaluation_split': 'official FashionMNIST test split; not used to train or select either model; Notebook 03 also uses it for descriptive objective diagnostics',
    'reference_data_split': 'official FashionMNIST training split',
    'evaluation_pairs': 64,
    'evaluation_seed': 2026,
    'trajectory_steps': 50,
    'bootstrap_resamples': 2000,
    'criteria': [
        'plausibility', 'semantic progression', 'target alignment and consistency',
        'non-target preservation', 'smoothness',
    ],
    'baseline_interpretation': {
        'identity': 'non-oracle; no transport',
        'global_shift': 'non-oracle; class means from training data',
        'linear_oracle': 'oracle-style; paired test target is used',
        'nn_linear_oracle': 'oracle-style; nearest training target is used',
    },
}

protocol_path = repo_root / 'assets' / 'evaluation' / 'evaluation_protocol.json'
protocol_path.parent.mkdir(parents=True, exist_ok=True)
protocol_path.write_text(json.dumps(PROTOCOL, indent=2) + '\n', encoding='utf-8')
print(f'Frozen protocol saved to {protocol_path}')